# Create Michael J. Fox Foundation Awards

Creates OpenAlex award rows from the official Michael J. Fox Foundation (MJFF) Funded Studies database.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/mjff_to_s3.py` to fetch MJFF's official pages, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data sources:**
- Official Funded Studies page: `https://www.michaeljfox.org/funded-studies`
- Official Drupal AJAX endpoint used by that page: `https://www.michaeljfox.org/ajax/funded-grants?page=N`
- Official grant detail pages under `/grant/{slug}`
- Official researcher profile pages under `/researcher/{slug}` for lead-profile affiliation/location enrichment

**S3 location:** `s3a://openalex-ingest/awards/mjff/mjff_funded_studies.parquet`

**Funder:** Michael J. Fox Foundation for Parkinson's Research, OpenAlex `F4320306136`, DOI `10.13039/100000864`.

**Source-authority note:** MJFF publishes project descriptions, researchers, program/year tags, keywords, and official landing pages. Full local validation on 2026-05-27 produced 2,655 rows (2000-2026), with 3 official pages lacking a funding year preserved as NULL and programless rows mapped to the explicit generic scheme `MJFF Funded Study`. MJFF does **not** publish per-grant award amounts in the public funded-study database, so amount/currency remain NULL with the same §6.7 waiver pattern used for HHMI/CIFAR/Damon Runyon/Schmidt-style fellowship/researcher archives.


## Step 1: Load Raw Parquet

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mjff_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/mjff/mjff_funded_studies.parquet`;

In [ ]:
%sql
-- Check raw row count. Local contractor validation on 2026-05-27 produced 2,655 rows.
SELECT COUNT(*) as total_mjff_funded_studies
FROM openalex.awards.mjff_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.mjff_raw;

In [ ]:
%sql
SELECT funder_award_id, display_name, source_year, source_program, lead_investigator_raw,
       lead_position_affiliation, landing_page_url
FROM openalex.awards.mjff_raw
ORDER BY TRY_CAST(source_year AS INT) DESC
LIMIT 10;

## Step 1.5: Source, Money, and Key Scans

In [ ]:
%sql
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'mjff_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|currency';

In [ ]:
%sql
-- Expected: amount/currency source columns exist only as intentional NULLs from the script.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS rows_with_title,
    COUNT(source_year) AS rows_with_year,
    COUNT(source_program) AS rows_with_program,
    COUNT(lead_investigator_raw) AS rows_with_lead,
    COUNT(lead_position_affiliation) AS rows_with_lead_affiliation,
    COUNT(description) AS rows_with_description,
    COUNT(amount) AS rows_with_amount,
    COUNT(currency) AS rows_with_currency,
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year
FROM openalex.awards.mjff_raw;

In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.mjff_raw;

In [ ]:
%sql
SELECT source_program, COUNT(*) AS rows
FROM openalex.awards.mjff_raw
GROUP BY source_program
ORDER BY rows DESC
LIMIT 25;

In [ ]:
%sql
SELECT source_year, COUNT(*) AS rows
FROM openalex.awards.mjff_raw
WHERE source_year IS NOT NULL
GROUP BY source_year
ORDER BY TRY_CAST(source_year AS INT) DESC
LIMIT 30;

In [ ]:
%sql
SELECT display_name, source_year, source_program, SUBSTR(description, 1, 240) AS description_preview, landing_page_url
FROM openalex.awards.mjff_raw
ORDER BY TRY_CAST(source_year AS INT) DESC
LIMIT 8;

## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for MJFF F4320306136'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320306136;

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320306136;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mjff_awards
USING delta
AS
WITH
funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306136
),
raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(source_year AS INT) AS parsed_year,
        TRY_TO_DATE(CONCAT(source_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(CONCAT(source_year, '-12-31'), 'yyyy-MM-dd') AS parsed_end_date,
        -- given_name / family_name are pre-split in the Python script via
        -- the split_name helper (ported from wolf_to_s3.py per
        -- how-to-add-a-funder-v2.md §2.4.1). Notebook only TRIM/NULLIFs.
        NULLIF(TRIM(lead_given_name), '') AS lead_given_name_clean,
        NULLIF(TRIM(lead_family_name), '') AS lead_family_name_clean,
        NULLIF(TRIM(
            CASE
                WHEN lead_position_affiliation RLIKE ' at ' THEN REGEXP_EXTRACT(lead_position_affiliation, ' at (.+)$', 1)
                ELSE lead_position_affiliation
            END
        ), '') AS lead_affiliation_name
    FROM openalex.awards.mjff_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,
        TRIM(g.display_name) as display_name,
        NULLIF(TRIM(g.description), '') as description,
        f.funder_id,
        g.native_award_id as funder_award_id,
        CAST(NULL AS DOUBLE) as amount,
        CAST(NULL AS STRING) as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,
        'research' as funding_type,
        COALESCE(NULLIF(TRIM(g.source_program), ''), 'MJFF Funded Study') as funder_scheme,
        'mjff_funded_studies' as provenance,
        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        g.parsed_year as start_year,
        g.parsed_year as end_year,
        CASE
            WHEN g.lead_family_name_clean IS NULL THEN NULL
            ELSE struct(
                g.lead_given_name_clean as given_name,
                g.lead_family_name_clean as family_name,
                CAST(NULL AS STRING) as orcid,
                g.parsed_start_date as role_start,
                struct(
                    g.lead_affiliation_name as name,
                    CAST(NULL AS STRING) as country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                ) as affiliation
            )
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN funder_resolved f
)
SELECT * FROM awards_transformed;

## Step 3: Delete Previous Source Rows and Insert Priority 129

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mjff_funded_studies' AND priority = 129;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    129 as priority
FROM openalex.awards.mjff_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) as total_mjff_awards
FROM openalex.awards.mjff_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.mjff_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_year,
       lead_investigator.given_name AS given_name,
       lead_investigator.family_name AS family_name,
       lead_investigator.affiliation.name AS affiliation,
       landing_page_url
FROM openalex.awards.mjff_awards
ORDER BY start_year DESC
LIMIT 10;

In [ ]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT id) AS distinct_openalex_ids,
       COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
       COUNT(*) - COUNT(DISTINCT funder_award_id) AS funder_award_id_collisions
FROM openalex.awards.mjff_awards;

In [ ]:
%sql
SELECT funder.display_name, funder_id, COUNT(*) AS cnt
FROM openalex.awards.mjff_awards
GROUP BY funder.display_name, funder_id;

In [ ]:
%sql
SELECT COUNT(*) as total,
       COUNT(description) as has_description,
       COUNT(amount) as has_amount,
       COUNT(currency) as has_currency,
       COUNT(lead_investigator.given_name) as has_lead_given,
       COUNT(lead_investigator.family_name) as has_lead_family,
       COUNT(lead_investigator.affiliation.name) as has_affiliation,
       COUNT(landing_page_url) as has_landing_page
FROM openalex.awards.mjff_awards;

In [ ]:
%sql
-- §6.7 waiver check: MJFF does not publish row-level amounts in the public funded-study database.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies
FROM openalex.awards.mjff_awards;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) AS rows
FROM openalex.awards.mjff_awards
GROUP BY funder_scheme
ORDER BY rows DESC
LIMIT 25;

In [ ]:
%sql
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.mjff_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 30;

In [ ]:
%sql
SELECT priority, provenance, COUNT(*) as cnt, COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mjff_funded_studies' AND priority = 129
GROUP BY priority, provenance;